# 🤖 AI Language Models & Prompt Engineering

In this session, we are going to learn how modern AI chatbots work and control one ourselves using Python. We will talk to a large language model running on our own hardware, learn what is happening inside it, and practice writing prompts that get better results.

By the end, you will understand what tokens are, how the AI uses memory, why the same question can give different answers, and how to design prompts that produce precise, useful outputs.

## Glossary

| Term | Description |
| --- | --- |
| Chain-of-Thought (CoT) | A prompting technique that asks the model to show its reasoning step by step before giving a final answer |
| Context Window | The maximum amount of text (measured in tokens) that a model can read at one time |
| Few-Shot Prompting | Providing multiple input/output examples in the prompt so the model copies the pattern |
| Hallucination | When an AI confidently states something that is false or made up |
| Inference | Running a trained AI model to get an answer (as opposed to training it) |
| Knowledge Cutoff | The date after which the model has no information, because training stopped |
| LLM | Large Language Model; a neural network trained on billions of sentences to predict text |
| One-Shot Prompting | Providing exactly one example in the prompt to guide the model's response |
| Parameter | A single tunable number inside a model; an 8B model has 8 billion of them |
| Prompt | The text you send to an AI model as your question or instruction |
| Prompt Engineering | The iterative, test-driven process of crafting prompts to get better outputs from an LLM |
| Role Prompting | Telling the model to adopt a specific persona using system instructions |
| Session | An independent conversation thread; each chat panel in this notebook has its own session |
| System Prompt | Instructions processed by the model before the conversation that define its role and behavior |
| Token | The smallest chunk of text the model processes; roughly ¾ of a word on average |
| Zero-Shot Prompting | Asking the model a question with no examples — it relies solely on pre-trained knowledge |

## 1. Setup

Let's start by loading in all the functionality we are going to need. Run this cell first before doing anything else. It checks that the required packages are installed and connects to the AI server running on the Strix board.

The `bootcamp_ai` module handles all the network communication for us. It exposes a simple `launch_chat()` function that opens an interactive chat panel, and a `prompt()` function for use directly in code.

In [1]:
import sys, subprocess

# Install any missing packages automatically
for pkg in ("requests", "ipywidgets"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg])

# Import our bootcamp AI helper
from bootcamp_ai import launch_chat, prompt, reset, list_models, use_model

# Check what models are available on the server
print("Models available on the Strix:", list_models())
print("\n ✅ Setup complete. You are ready to go!")

Models available on the Strix: ['Qwen3-Coder-30B-A3B-Instruct-GGUF', 'Qwen3-VL-8B-Instruct-GGUF', 'Qwen3.6-27B-GGUF']

 ✅ Setup complete. You are ready to go!


**Note:** If you see an empty list or a connection error, the AI server on the Strix may not be running. Ask an instructor to check that Lemonade is running on the Strix board before continuing.

## 2. What Is an LLM?

**LLM** stands for **Large Language Model**. It is a type of neural network that has been trained on an enormous amount of text billions of web pages, books, articles, and code. The result is a model that can read and write human language convincingly.

### How it works — one trick, repeated many times

At its core, an LLM does exactly one thing: **it predicts the most likely next token** given everything it has seen so far. It does this over and over, one token at a time, until the answer is complete.

For example, given the text `"The closest planet to the Sun is"`, the model produces a probability for every token it knows:

```
Mercury   ████████████████  78%
Venus     ███               12%
Mars      █                  4%
...       (all other tokens combined) 6%
```

It samples from that distribution, picks `"Mercury"`, adds it to the text, and runs again. After ~30 picks you have a full sentence. After a few hundred you have a paragraph.

### Why does that work so well?

The "Quick Helper" model on your Strix has **8 billion parameters**. Each parameter is a number that was adjusted during training to make predictions better. After being trained on enough text, those 8 billion numbers encode an enormous amount of knowledge about how language and the world works.

<div style="background:#1a2230;border:1px solid #2f3b4d;border-radius:12px;padding:18px;font-family:sans-serif;"><div style="font-size:19px;font-weight:600;color:#9db4d0;margin-bottom:12px;">&#129504; The LLM predicts ONE word at a time</div><div style="font-family:monospace;font-size:14px;color:#8fa3bf;margin-bottom:12px;">"The Kria KV260 is a powerful ___"</div><table style="border-collapse:collapse;width:100%;"><tr><td style="width:110px;font-family:monospace;font-size:13px;color:#8fa3bf;padding:3px 8px 3px 0;">FPGA</td><td style="padding:3px 0;"><div style="background:#5a7a9a;height:18px;width:68%;border-radius:4px;"></div></td><td style="width:46px;font-family:monospace;font-size:13px;color:#8fa3bf;font-weight:600;padding-left:8px;">68%</td></tr><tr><td style="font-family:monospace;font-size:13px;color:#8fa3bf;padding:3px 8px 3px 0;">board</td><td style="padding:3px 0;"><div style="background:#4a6580;height:18px;width:21%;border-radius:4px;"></div></td><td style="font-family:monospace;font-size:13px;color:#8fa3bf;font-weight:600;padding-left:8px;">21%</td></tr><tr><td style="font-family:monospace;font-size:13px;color:#8fa3bf;padding:3px 8px 3px 0;">chip</td><td style="padding:3px 0;"><div style="background:#3d5468;height:18px;width:8%;border-radius:4px;"></div></td><td style="font-family:monospace;font-size:13px;color:#8fa3bf;font-weight:600;padding-left:8px;">8%</td></tr></table><div style="color:#7a8ba3;font-size:13px;margin-top:10px;">&#128257; The model picks the top word, adds it, and predicts again &mdash; about 30 times to make a full sentence.</div><div style="background:#232d3d;border:1px solid #354258;border-radius:8px;padding:10px 14px;margin-top:10px;font-size:13px;color:#9db4d0;"><b>&#127760; Real life:</b> When you type a search into Google or ask ChatGPT a question, this exact loop is running &mdash; guessing the next word over and over. Your KV260 Strix does the same thing, accelerated by the DPU.</div></div>

### What this means for you

- The model is **not looking things up** in a database. It is generating answers from patterns it memorized during training.
- The same question can produce **different answers** each time, because there is randomness in the sampling step, this is controlled by a setting called **temperature**.
- The model can be **confidently wrong** it predicts likely-sounding text, not necessarily true text.

In [2]:
# ===== GAME: Next-Word Predictor (run me, then click!) =====
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import random

_DB = {
    "the cat sat on the": [("mat",61),("floor",18),("roof",9),("sofa",7),("bed",5)],
    "to load a dpu overlay you call": [("DpuOverlay()",72),("overlay.load()",14),("import pynq",6),("dpu.run()",5),("load()",3)],
    "the kria kv260 is a powerful": [("FPGA",64),("board",21),("chip",9),("computer",4),("tool",2)],
    "python is a": [("programming",45),("popular",22),("high-level",18),("versatile",10),("great",5)],
    "my favorite food is": [("pizza",40),("pasta",25),("sushi",18),("tacos",12),("ice",5)],
}

def _bars(probs):
    cols = ["#4ade80","#38bdf8","#fbbf24","#a78bfa","#fb923c"]
    out = ""
    for i,(t,p) in enumerate(probs):
        out += (f"<div style='display:flex;align-items:center;gap:8px;margin:5px 0;'>"
                f"<div style='width:130px;text-align:right;font-family:monospace;font-size:13px;color:#e7edf5;'>{t}</div>"
                f"<div style='background:{cols[i%5]};height:20px;width:{max(8,int(p*2.6))}%;border-radius:4px;'></div>"
                f"<div style='color:{cols[i%5]};font-weight:700;font-size:12px;'>{p}%</div></div>")
    return out

def _lookup(text):
    t = text.strip().lower().rstrip(".!?,")
    for k in _DB:
        if t.endswith(k):
            return _DB[k]
    pcts = sorted([random.randint(5,50) for _ in range(5)], reverse=True)
    tot = sum(pcts)
    return list(zip(["the","is","a","and","to"], [round(p/tot*100) for p in pcts]))

_inp = widgets.Text(value="The Kria KV260 is a powerful",
                    layout=widgets.Layout(width="480px"))
_btn = widgets.Button(description="Predict next word!", button_style="primary",
                      layout=widgets.Layout(width="170px", height="36px"))
_out = widgets.Output()

def _go(_):
    probs = _lookup(_inp.value)
    with _out:
        clear_output(wait=True)
        display(HTML(
            f"<div style='font-family:sans-serif;background:#0f1419;border-radius:10px;padding:16px;margin-top:8px;'>"
            f"<div style='color:#8a97ab;font-size:12px;margin-bottom:8px;'>Your text: "
            f"<em style='color:#e7edf5;'>\"{_inp.value}\"</em></div>{_bars(probs)}"
            f"<div style='margin-top:10px;color:#4ade80;font-size:12px;'>The model picks "
            f"<b>{probs[0][0]}</b>, adds it, and predicts again!</div></div>"))

_btn.on_click(_go)
display(HTML("<div style='font-family:sans-serif;font-size:16px;font-weight:700;color:#1e2a4a;'>"
             "&#127918; GAME: Be the AI &mdash; Predict the Next Word</div>"
             "<div style='font-size:13px;color:#64748b;margin-bottom:8px;'>Type the start of a "
             "sentence and see the model's top 5 guesses. Try the examples in the boxes above!</div>"))
display(widgets.HBox([_inp, _btn]))
display(_out)
_go(None)


Output()

In [3]:
# Open a chat panel for Part 2.
# Each launch_chat() creates its own independent memory session.
launch_chat()

bootcamp_ai v2026.07.10-10 starting (no Gradio)…


'http://192.168.101.8:8899'

**Try these prompts in the chat panel above, one at a time:**

1. `Explain what an LLM is in two sentences, for a high school student.`
2. `Finish this sentence: If your Jupyter notebook kernel crashes, the first thing you should`
3. `Write one sentence about robots.` — then click **✦ New chat** and type the exact same thing again. Did you get the same words? Why not?

> **What to notice on prompt 3:** Even though the input was identical, the output changed. The model samples from a probability distribution rather than always picking the top word. 
The setting that controls this is called **temperature** — at temperature 0 it always picks the same word; at temperature 1 it varies. Your Strix runs at around 0.7.

## 3. Tokens — How the Model Reads

The model does not read words the way you do. It breaks all text — your question and its answer — into small chunks called **tokens**. A token can be:

- A complete common word: `"cat"`, `"is"`, `"the"`
- Part of a longer word: `"bas"` + `"ket"` + `"ball"`
- A punctuation mark: `"."`, `"!"`, `":"`
- A space plus a word: `" hello"`

A rough rule of thumb is **1 token ≈ ¾ of an English word**. So 100 words ≈ 133 tokens.

### Why tokens matter

Tokens are the fundamental unit of everything the model does:

| What | Measured in tokens |
| --- | --- |
| Context window size | e.g. 128,000 tokens max |
| Speed | tokens per second (tok/s) |
| API cost | billed per token (on paid services) |

The timing line at the bottom of each chat answer ("answered in X.Xs") reflects how many tokens the model generated and how fast.

<div style="background:#141a24;border:1px solid #2f3b4d;border-radius:12px;padding:18px;font-family:sans-serif;"><div style="font-size:19px;font-weight:700;color:#38bdf8;margin-bottom:12px;">&#129513; The AI reads TOKENS, not words</div><div style="font-size:14px;color:#cbd5e1;margin-bottom:12px;">The word <b style="color:#fff;">"DpuOverlay"</b> is split into 4 tokens:</div><div style="margin-bottom:12px;"><span style="display:inline-block;background:#3b5bdb;color:#fff;font-family:monospace;font-weight:700;font-size:15px;padding:6px 14px;border-radius:16px;margin-right:6px;">Dp</span><span style="display:inline-block;background:#db2777;color:#fff;font-family:monospace;font-weight:700;font-size:15px;padding:6px 14px;border-radius:16px;margin-right:6px;">u</span><span style="display:inline-block;background:#059688;color:#fff;font-family:monospace;font-weight:700;font-size:15px;padding:6px 14px;border-radius:16px;margin-right:6px;">Over</span><span style="display:inline-block;background:#ea580c;color:#fff;font-family:monospace;font-weight:700;font-size:15px;padding:6px 14px;border-radius:16px;">lay</span></div><div style="font-size:13px;color:#94a3b8;">Rule of thumb: 1 token &#8776; three-quarters of a word &nbsp;&middot;&nbsp; 100 words &#8776; 133 tokens</div><div style="background:#10233b;border:1px solid #1e4976;border-radius:8px;padding:10px 14px;margin-top:10px;font-size:13px;color:#cfe3ff;"><b>&#127760; Real life:</b> Paid AI services (like the ChatGPT API) charge you <b>per token</b>. A long chatbot conversation literally costs more money because more tokens are processed.</div></div>

In [4]:
# ===== GAME: Live Token Counter (type and watch!) =====
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import re

def _tok(text):
    if not text: return []
    raw = re.findall(r"[A-Z][a-z]*|[a-z]+|[0-9]+|[^a-zA-Z0-9\s]", text)
    out = []
    for r in raw:
        if len(r) > 5:
            m = len(r)//2; out += [r[:m], r[m:]]
        else: out.append(r)
    return out or [text]

_cols = ["#3b5bdb","#db2777","#059688","#ea580c","#7c3aed","#16a34a","#0284c7"]
_ta = widgets.Textarea(value="I love building robots on the KV260!",
                       layout=widgets.Layout(width="600px", height="55px"))
_o = widgets.Output()

def _upd(_):
    toks = _tok(_ta.value)
    chips = "".join(
        f"<span style='display:inline-block;background:{_cols[i%7]};color:#fff;"
        f"padding:4px 10px;border-radius:14px;font-size:13px;font-weight:600;"
        f"margin:3px;font-family:monospace;'>{t}</span>" for i,t in enumerate(toks))
    with _o:
        clear_output(wait=True)
        display(HTML(
            f"<div style='font-family:sans-serif;margin-top:8px;'>"
            f"<div style='display:flex;gap:14px;margin-bottom:12px;'>"
            f"<div style='background:#eef1ff;border-radius:8px;padding:8px 16px;text-align:center;'>"
            f"<div style='font-size:24px;font-weight:700;color:#3b5bdb;'>{len(toks)}</div>"
            f"<div style='font-size:11px;color:#64748b;'>tokens</div></div>"
            f"<div style='background:#fce7f3;border-radius:8px;padding:8px 16px;text-align:center;'>"
            f"<div style='font-size:24px;font-weight:700;color:#db2777;'>{len(_ta.value)}</div>"
            f"<div style='font-size:11px;color:#64748b;'>characters</div></div>"
            f"<div style='background:#dcfce7;border-radius:8px;padding:8px 16px;text-align:center;'>"
            f"<div style='font-size:24px;font-weight:700;color:#16a34a;'>{len(_ta.value.split())}</div>"
            f"<div style='font-size:11px;color:#64748b;'>words</div></div></div>"
            f"<div>{chips}</div>"
            f"<div style='margin-top:8px;font-size:11px;color:#94a3b8;'>Each chip = 1 token. "
            f"Long words split into pieces!</div></div>"))

_ta.observe(_upd, names="value")
display(HTML("<div style='font-family:sans-serif;font-size:16px;font-weight:700;color:#1e2a4a;'>"
             "&#127918; GAME: Live Token Counter</div>"
             "<div style='font-size:13px;color:#64748b;margin-bottom:8px;'>Type anything and watch it "
             "break into tokens as you type. Try a really long word!</div>"))
display(_ta); display(_o); _upd(None)


Textarea(value='I love building robots on the KV260!', layout=Layout(height='55px', width='600px'))

Output()

In [5]:
# A simple token estimator based on character count.

def estimate_tokens(text):
    """Estimate the number of tokens in a piece of text.
    Rule of thumb: 1 token ≈ 4 characters of English text."""
    return max(1, round(len(text) / 4))

examples = [
    "cat",
    "I love building robots with Python!",
    "Explain how an FPGA works in three bullet points for a high school student."
]

print(f"{'Text':<55} {'Est. tokens':>12}")
print("-" * 70)
for t in examples:
    print(f"{repr(t):<55} {estimate_tokens(t):>12}")

Text                                                     Est. tokens
----------------------------------------------------------------------
'cat'                                                              1
'I love building robots with Python!'                              9
'Explain how an FPGA works in three bullet points for a high school student.'           19


In [6]:
# Open a chat panel for Part 3.
launch_chat()

bootcamp_ai v2026.07.10-10 starting (no Gradio)…


'http://192.168.101.8:8899'

**Try these prompts in the chat panel above:**

1. `Break the word 'basketball' into token-sized pieces and show each piece.`
2. `Roughly how many tokens is the sentence "I love building robots with Python"?`
3. Ask for a 1-sentence answer on any topic, then ask for a 10-sentence answer on the same topic. Watch the timing line — more tokens = more time.

> **Note:** The chat panel shows "answered in X.Xs (first word Y.Ys)" under each reply. The "first word" time is how long until tokens started arriving. Use the model dropdown to switch between Quick Helper (8B) and Code Helper (30B) and compare the tok/s — bigger model, slower speed.

## 4. Context and Memory

Here is one of the most important and surprising facts about language models:

**The model has no built-in memory between calls.** Every time you send a message, the model starts completely fresh.

So how does a chatbot remember what you said earlier? The answer is that *we* — the code on our side — keep a list of every message sent and received, and re-send the entire list every time you ask a new question. That list is called the **context**.

```
Turn 1:  We send  → ["User: My name is Nikhitha."]
         We get   → "Nice to meet you, Nikhitha!"

Turn 2:  We send  → ["User: My name is Nikhitha.",
                      "AI: Nice to meet you, Nikhitha!",
                      "User: What is my name?"]   ← full history every time
         We get   → "Your name is Nikhitha!"
```

If we delete the first message from the list, the model genuinely cannot answer — the information is gone.

### The context window limit

The context has a maximum size in tokens. When it fills up, the oldest messages are dropped and the AI truly forgets them — like a whiteboard that runs out of space. The `bootcamp_ai` module manages this with a sliding window that keeps the most recent turns within a character budget.

**Secret: the AI has no built-in memory.** The chat panel keeps a list of your conversation and re-sends it each turn. That list is the **context** — the AI's short-term memory.

> **🐠 The Goldfish Problem:** The context has a size limit (in tokens). Very long chats push the *oldest* messages out, and the AI forgets them.

> **&#127760; Real life:** This is why a long ChatGPT chat sometimes "forgets" what you said at the start &mdash; the oldest messages fall out of the context window to make room, like a whiteboard running out of space.

In [7]:
# ===== GAME: Memory ON vs OFF (asks the real AI!) =====
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

_nm = widgets.Text(value="Maya", description="Your name:",
                   style={"description_width":"80px"}, layout=widgets.Layout(width="300px"))
_b = widgets.Button(description="Ask both: What is my name?", button_style="primary",
                    layout=widgets.Layout(width="250px", height="38px"))
_o = widgets.Output()

def _go(_):
    name = _nm.value.strip() or "Maya"
    with _o:
        clear_output(wait=True); print("Asking the AI both ways...")
    reset()
    prompt(f"My name is {name} and I'm at the PYNQ Bootcamp.", enforce_cooldown=False)
    on = prompt("What is my name?", enforce_cooldown=False)
    reset()
    off = prompt("What is my name?", enforce_cooldown=False)
    with _o:
        clear_output(wait=True)
        display(HTML(
            f"<div style='font-family:sans-serif;display:flex;gap:12px;margin-top:8px;'>"
            f"<div style='flex:1;background:#dcfce7;border:2px solid #22c55e;border-radius:10px;padding:12px;'>"
            f"<div style='font-weight:700;color:#15803d;font-size:12px;margin-bottom:6px;'>&#9989; MEMORY ON</div>"
            f"<div style='font-size:12px;color:#374151;'>{on[:180]}</div></div>"
            f"<div style='flex:1;background:#fee2e2;border:2px solid #ef4444;border-radius:10px;padding:12px;'>"
            f"<div style='font-weight:700;color:#b91c1c;font-size:12px;margin-bottom:6px;'>&#10060; MEMORY OFF</div>"
            f"<div style='font-size:12px;color:#374151;'>{off[:180]}</div></div></div>"
            f"<div style='font-family:sans-serif;margin-top:10px;background:#fef9c3;border:1px solid #fbbf24;"
            f"border-radius:8px;padding:10px;font-size:12px;color:#78350f;'>&#128161; Same question, same AI &mdash; "
            f"the only difference is whether we sent the history list. That list <b>is</b> the memory!</div>"))

_b.on_click(_go)
display(HTML("<div style='font-family:sans-serif;font-size:16px;font-weight:700;color:#1e2a4a;'>"
             "&#127918; GAME: Memory ON vs OFF</div>"
             "<div style='font-size:13px;color:#64748b;margin-bottom:8px;'>Enter your name, then watch the "
             "AI answer the SAME question two ways &mdash; with and without memory.</div>"))
display(widgets.HBox([_nm, _b])); display(_o)


Output()

In [8]:
# Explore memory directly in code — no chat panel needed.

reset()  # start a completely fresh conversation

# Turn 1: give the model some information about you
r1 = prompt("My name is Nikhitha and my favorite subject is robotics.", enforce_cooldown=False)
print("AI:", r1)
print("-" * 50)


AI: Hi Nikhitha! 👋  
That’s awesome — robotics is so cool and super fun to learn! 🤖  
If you ever want to work with PYNQ, cameras, sensors, or even make your own robot projects, I’m here to help.  
What kind of robotics project are you working on right now? 😊
--------------------------------------------------


In [9]:
# Turn 2: ask it to recall — it should remember because the history is re-sent
r2 = prompt("What is my name and favorite subject?", enforce_cooldown=False)
print("AI:", r2)

AI: Your name is **Nikhitha** 🎉  
And your favorite subject is **robotics**! 🤖  
So proud you’re into it — robots are awesome to build and program!  
Let me know if you want help with anything robotics-related. 😊


In [10]:
# Now wipe the memory and ask again — it cannot know.

reset()  # empties the history list completely

r3 = prompt("What is my name?", enforce_cooldown=False)
print("AI:", r3)
print("\n It can't know — the context was cleared!")

AI: I don’t know your name — you haven’t told me yet! 😊  
You can tell me your name anytime, and I’ll remember it for our PYNQ coding adventures.  
What’s your project today? I’m here to help! 🤖💻

 It can't know — the context was cleared!


In [8]:
# Open a chat panel for Part 4.
launch_chat()

bootcamp_ai v2026.07.10-10 starting (no Gradio)…


'http://192.168.101.8:8899'

**Try this exact sequence in the chat panel:**

**Step 1 — show that it remembers:**
1. `My name is [your name] and my PYNQ project is about [your project].`
2. `What is my name and what am I building?` — it remembers, because both messages are in the context.

**Step 2 — erase the memory:**
3. Click **✦ New chat** at the top of the panel.
4. `What is my name?` — it has no idea. The context list was emptied.

> **Key takeaway:** Memory in AI is not magic — it is a Python list that gets re-sent each turn. The "New chat" button deletes that list.

## 5. Prompt Engineering

A **prompt** is the text you send to an AI model. **Prompt engineering** is the practice of crafting and refining prompts to elicit better, more accurate, and more useful responses from the model.

According to Google Cloud, prompt engineering is an *iterative, test-driven process* — you write a prompt, evaluate the output, refine the wording, and repeat. The structure of a prompt (ordering, labeling, use of delimiters) matters just as much as its content.

### The anatomy of a strong prompt

A well-engineered prompt typically combines some of these ingredients:

| Ingredient | Purpose | Example |
| --- | --- | --- |
| **Role** | Sets the model's persona, tone, and expertise | `You are an experienced hardware engineer.` |
| **Task** | States exactly what you want | `Explain how an FPGA differs from a CPU.` |
| **Format** | Specifies how the answer should be structured | `Give your answer in exactly 3 bullet points.` |
| **Audience** | Adjusts vocabulary and detail level | `Write for a high school student with no prior experience.` |
| **Constraints** | Sets limits or rules | `Keep your answer under 100 words.` |

You do not need all five every time, but more direction usually means a better answer.


<div style="background:#141a24;border:1px solid #2f3b4d;border-radius:12px;padding:18px;font-family:sans-serif;"><div style="font-size:19px;font-weight:700;color:#38bdf8;margin-bottom:12px;">&#9997; The 5 ingredients of a strong prompt</div><table style="border-collapse:separate;border-spacing:8px 0;width:100%;table-layout:fixed;"><tr><td style="background:#1e2733;border-radius:10px;padding:12px 8px;text-align:center;vertical-align:top;"><div style="font-size:24px;">&#127917;</div><div style="font-size:13px;font-weight:700;color:#60a5fa;margin:5px 0 2px;">Role</div><div style="font-size:10px;color:#94a3b8;">who to be</div></td><td style="background:#1e2733;border-radius:10px;padding:12px 8px;text-align:center;vertical-align:top;"><div style="font-size:24px;">&#128221;</div><div style="font-size:13px;font-weight:700;color:#f472b6;margin:5px 0 2px;">Task</div><div style="font-size:10px;color:#94a3b8;">what to do</div></td><td style="background:#1e2733;border-radius:10px;padding:12px 8px;text-align:center;vertical-align:top;"><div style="font-size:24px;">&#128203;</div><div style="font-size:13px;font-weight:700;color:#4ade80;margin:5px 0 2px;">Format</div><div style="font-size:10px;color:#94a3b8;">how to lay out</div></td><td style="background:#1e2733;border-radius:10px;padding:12px 8px;text-align:center;vertical-align:top;"><div style="font-size:24px;">&#128101;</div><div style="font-size:13px;font-weight:700;color:#fbbf24;margin:5px 0 2px;">Audience</div><div style="font-size:10px;color:#94a3b8;">who it is for</div></td><td style="background:#1e2733;border-radius:10px;padding:12px 8px;text-align:center;vertical-align:top;"><div style="font-size:24px;">&#9939;</div><div style="font-size:13px;font-weight:700;color:#fb923c;margin:5px 0 2px;">Limits</div><div style="font-size:10px;color:#94a3b8;">rules</div></td></tr></table><div style="background:#10233b;border:1px solid #1e4976;border-radius:8px;padding:10px 14px;margin-top:10px;font-size:13px;color:#cfe3ff;"><b>&#127760; Real life:</b> Prompt engineering is now a real job title &mdash; companies pay people to write prompts that get reliable answers from AI. The skill you learn here is genuinely used in industry.</div></div>

In [12]:
# ===== GAME: Prompt Duel (edit both, then RUN!) =====
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

_w = widgets.Textarea(value="Tell me about FPGAs.", description="Weak:",
    layout=widgets.Layout(width="48%", height="75px"), style={"description_width":"55px"})
_s = widgets.Textarea(
    value="You are a hardware engineer. Explain what an FPGA is and give 3 examples, for a high school student.",
    description="Strong:", layout=widgets.Layout(width="48%", height="75px"),
    style={"description_width":"55px"})
_b = widgets.Button(description="Run the Duel!", button_style="primary",
                    layout=widgets.Layout(width="160px", height="38px"))
_l = widgets.Output(layout=widgets.Layout(width="48%", border="2px solid #ef4444", padding="8px"))
_r = widgets.Output(layout=widgets.Layout(width="48%", border="2px solid #22c55e", padding="8px"))
_why = widgets.Output()

def _an(bef, aft):
    b, a = bef.lower(), aft.lower(); t = []
    if "you are" not in b and "you are" in a: t.append("&#127917; added a ROLE")
    if not any(w in b for w in ("bullet","step","example","point")) and any(w in a for w in ("bullet","step","example","point")): t.append("&#128203; added a FORMAT")
    if not any(w in b for w in ("grader","student","for a")) and any(w in a for w in ("grader","student","for a")): t.append("&#128101; named an AUDIENCE")
    if len(aft) > len(bef)*1.3: t.append("&#128269; more DETAIL")
    return t or ["&#128221; more specific"]

def _go(_):
    for o in (_l,_r,_why): o.clear_output()
    with _l: print("thinking...")
    with _r: print("thinking...")
    reset(); wa = prompt(_w.value, enforce_cooldown=False)
    reset(); sa = prompt(_s.value, enforce_cooldown=False)
    _l.clear_output(); _r.clear_output()
    with _l: print("WEAK ANSWER\n" + "-"*24 + "\n" + wa)
    with _r: print("STRONG ANSWER\n" + "-"*24 + "\n" + sa)
    with _why:
        clear_output(wait=True)
        display(HTML("<div style='font-family:sans-serif;background:#fffbf0;border:1.5px solid #f59f00;"
                     "border-radius:8px;padding:12px;margin-top:10px;'><b style='color:#6b4f00;'>"
                     "&#128161; What made the strong prompt win:</b>&nbsp;&nbsp;"
                     + "&nbsp;&bull;&nbsp;".join(_an(_w.value, _s.value)) + "</div>"))

_b.on_click(_go)
display(HTML("<div style='font-family:sans-serif;font-size:16px;font-weight:700;color:#1e2a4a;'>"
             "&#127918; GAME: Prompt Duel</div>"
             "<div style='font-size:13px;color:#64748b;margin-bottom:8px;'>Edit the weak and strong prompts, "
             "click Run, and see who wins!</div>"))
display(widgets.HBox([_w, _s])); display(_b); display(widgets.HBox([_l, _r])); display(_why)


Button(button_style='primary', description='Run the Duel!', layout=Layout(height='38px', width='160px'), style…

Output()

### 5.1 Zero-Shot Prompting

Zero-shot prompting means giving the model a direct instruction or question with **no examples** at all. The model relies entirely on its pre-trained knowledge to respond.

This is the most common way people use AI — just ask a question and get an answer. It works well for common, well-defined tasks like summarisation, translation, brainstorming, or general questions.

```
# Zero-shot example
Summarise the main causes of World War One in three bullet points.
```

**When to use it:** Any general task where the model has enough training knowledge to do it well without needing examples.

In [9]:
# Zero-shot: just ask, no examples provided.
reset()
result = prompt(
    "Summarise what an FPGA is and give one real-world use case. Keep it under 60 words.",
    enforce_cooldown=False
)
print(result)

An FPGA (Field-Programmable Gate Array) is a chip you can program to do specific tasks, like a super-flexible circuit. Real-world use: In self-driving cars, FPGAs process sensor data super fast to help the car make split-second decisions. 🚗⚡


### 5.2 One-Shot Prompting

One-shot prompting provides **exactly one example** of the input/output pattern you want before asking your real question. A single example gives the model a concrete reference point for the format, tone, or style you expect.

```
# One-shot example
Convert a technical term into a simple one-sentence definition.

Term: FPGA
Definition: A chip you can reprogram to act as custom hardware for a specific task.

Term: Tensor
Definition:
```

The model sees the `Term → Definition` pattern from one example and continues it.

**When to use it:** When you need a specific output format and zero-shot gives inconsistent results, but you only have one good example to provide.

In [10]:
# One-shot: one example to set the pattern, then the real question.
reset()
one_shot_prompt = """Convert a technical term into a simple one-sentence definition.

Term: FPGA
Definition: A chip you can reprogram to act as custom hardware for a specific task.

Term: Tensor
Definition:"""

print(prompt(one_shot_prompt, enforce_cooldown=False))

A tensor is like a multi-dimensional array that holds data for things like images or machine learning models — think of it as a flexible container for numbers used by computers to do smart stuff. 🧠📦


### 5.3 Few-Shot Prompting

Few-shot prompting provides **multiple examples** (typically 2–5) of the pattern you want. The model identifies the relationship between inputs and outputs from those examples and applies the same pattern to the new input.

According to Google Cloud: *"When you provide a model with examples, you help control aspects of the model's response such as formatting, phrasing, scope, and overall patterns."*

Important notes from Google Cloud research:
- Too **few** examples can be ineffective at changing model behaviour.
- Too **many** examples can cause the model to **overfit** — it copies the examples so closely it loses flexibility.
- Effective few-shot prompts combine **clear instructions** with **specific and varied examples**.

```
# Few-shot example — classify sensor readings
Classify the sensor reading as NORMAL, WARNING, or CRITICAL.

Reading: Temperature 22°C  → NORMAL
Reading: Temperature 78°C  → WARNING
Reading: Temperature 105°C → CRITICAL
Reading: Temperature 91°C  →
```

**When to use it:** Custom classification, unusual output formats, consistent style enforcement, or any task where zero-shot gives variable results.

In [11]:
# Few-shot: three examples teach the model the classification pattern.
reset()
few_shot_prompt = """Classify the sensor reading as NORMAL, WARNING, or CRITICAL.

Reading: Temperature 22°C  -> NORMAL
Reading: Temperature 78°C  -> WARNING
Reading: Temperature 105°C -> CRITICAL
Reading: Temperature 91°C  ->"""

print(prompt(few_shot_prompt, enforce_cooldown=False))

WARNING

*(Above 78°C is WARNING, and 91°C is still under the CRITICAL threshold of 105°C.)* 🌡️


In [12]:
# Compare: zero-shot vs few-shot on the same classification task.
# Zero-shot gives the right answer too, but few-shot forces the exact format.

reset()
zero = prompt("Is a CPU temperature of 91°C normal, a warning, or critical?", enforce_cooldown=False)
print("ZERO-SHOT:", zero)

print()

reset()
few = prompt(few_shot_prompt, enforce_cooldown=False)  # reuse the few-shot prompt from above
print("FEW-SHOT:", few)

# Notice: zero-shot writes a sentence; few-shot produces exactly "WARNING" or "CRITICAL" matching the pattern.

ZERO-SHOT: A CPU temperature of **91°C is a warning**, not critical — but it’s still *too hot* and should be addressed.

👉 **What’s normal?**  
- Idle: 40–60°C  
- Under load (e.g., running code, video, games): 70–85°C is ideal  
- **91°C = warning zone** — your CPU is overheating and may throttle performance or risk damage if left unchecked.

✅ **What to do?**  
- Check cooling (fan, heatsink, thermal paste)  
- Clean dust from fan/heatsink  
- Ensure good airflow in your case  
- If using a Kria/KV260 or PYNQ board, make sure the fan is running or you’re not overloading the CPU

⚠️ If it hits 95°C+ regularly, it’s *critical*.  
🛑 If it’s stuck at 100°C+, immediate action is needed — your CPU could be damaged.

💡 Pro tip: Monitor temperature with tools like `vcgencmd measure_temp` (Raspberry Pi) or `sensors` (Linux).  
Let me know if you want help checking it!

🌡️ Keep it cool — your code runs better when your CPU does! 🤖

> *Note: If you’re using a Kria/KV260 or PYNQ board, let me kn

### 5.4 Chain-of-Thought (CoT) Prompting

Chain-of-Thought prompting asks the model to **explicitly show its reasoning steps** before arriving at a final answer. Instead of jumping to the conclusion, the model walks through intermediate steps — like showing your work in a maths exam.

This technique significantly improves performance on **complex reasoning tasks** — multi-step maths, logic puzzles, debugging, planning — because writing out the steps forces the model to stay consistent rather than guessing.

There are two variants:

| Variant | How it works |
| --- | --- |
| **Standard CoT** | Provide examples that show reasoning steps, then ask the real question |
| **Zero-Shot CoT** | Just add `"Think step by step."` or `"Explain your reasoning first."` to any prompt |

```
# Zero-shot CoT — just add an instruction to reason aloud
A Raspberry Pi runs at 1.8 GHz. A PYNQ board's DPU can run
100 operations per clock cycle. How many operations per second
can the DPU perform? Think step by step.
```

In [ ]:
# Without Chain-of-Thought: the model may give a direct answer that skips reasoning.
reset()
without_cot = prompt(
    "A loop runs 500 times. Each iteration takes 3ms. How long does the loop take in total?",
    enforce_cooldown=False
)
print("WITHOUT CoT:")
print(without_cot)

In [ ]:
# WITH Chain-of-Thought: ask the model to show its work.
# "Think step by step" is the simplest CoT trigger — it works surprisingly well.
reset()
with_cot = prompt(
    "A loop runs 500 times. Each iteration takes 3ms. "
    "How long does the loop take in total? Think step by step.",
    enforce_cooldown=False
)
print("WITH CoT:")
print(with_cot)

In [ ]:
# CoT is especially powerful for debugging.
# Ask the model to reason through the bug before suggesting a fix.
reset()
buggy_code = """
def average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)

print(average([]))  # crashes!
"""

debug_prompt = (
    f"Here is a Python function that crashes:\n{buggy_code}\n"
    "Explain step by step WHY it crashes, then suggest the fix."
)

print(prompt(debug_prompt, enforce_cooldown=False))

### 5.5 Role Prompting

Role prompting tells the model to **adopt a specific persona** before answering. You define the role in the prompt (or in a system prompt that the model reads before the conversation starts).

According to Google Cloud: *"If you are going to ask the model to act in a specific role, make sure that role is defined in the system instructions. System instructions tell the model how you want it to behave and respond — including the role or persona, contextual information, and formatting instructions."*

Role prompting controls:
- **Tone** — formal vs. casual, encouraging vs. critical
- **Vocabulary** — technical jargon vs. plain language
- **Perspective** — how the model frames its answers
- **Behaviour** — what the model will and won't do

The role stays active for the whole conversation because the system instructions live in the context window.

In [ ]:
# Role prompting: same question, completely different voices.
# The only change is the role at the start of each prompt.

question = "Explain what a neural network is."

roles = [
    ("a PhD researcher presenting at a conference",       "RESEARCHER"),
    ("a friendly teacher explaining to a 12-year-old",   "TEACHER   "),
    ("an excited sports commentator",                     "SPORTS    "),
]

for role_desc, label in roles:
    reset()
    answer = prompt(f"You are {role_desc}. {question} Keep it to 2 sentences.",
                    enforce_cooldown=False)
    print(f"[{label}] {answer}")
    print()

### 5.6 Prompt Engineering Best Practices

The following guidelines come from Google Cloud's prompt engineering research:

| Practice | Why it helps |
| --- | --- |
| **Be specific** | Vague prompts produce vague answers. State exactly what you need. |
| **Define the output format** | Tell the model how to structure the answer (bullet list, table, code block, etc.) |
| **Use delimiters** | Separate instructions from content using `---`, triple quotes, or XML-style tags |
| **Label your sections** | Use headings like `Input:` and `Output:` to help the model parse complex prompts |
| **Iterate** | Treat prompting as a test-driven loop — evaluate the output and refine the wording |
| **Order matters** | The model pays more attention to content at the start and end of long prompts |
| **Add constraints** | "Keep under 100 words" or "do not use jargon" prevents unwanted behaviours |
| **Ask for self-critique** | "What might be wrong with the answer you just gave?" can catch errors |

In [ ]:
# Best practices in action: a well-structured prompt using delimiters and labels.

reset()
structured_prompt = """
You are a PYNQ board technical assistant.

Task: Review the code below and identify any bugs or improvements.

Output format:
- BUGS: list any errors that would cause a crash
- IMPROVEMENTS: list style or performance suggestions
- FIXED CODE: show the corrected version

---
Code to review:
```python
def read_sensor(pin):
    values = []
    for i in range(10):
        values.append(pin.read)
    return sum(values) / 10
```
---
"""

print(prompt(structured_prompt, enforce_cooldown=False))

In [13]:
# Open a chat panel to try all the techniques interactively.
launch_chat()

bootcamp_ai v2026.07.10-10 starting (no Gradio)…


'http://192.168.101.8:8899'

**Exercises — try each prompting technique in the chat panel:**

**Zero-shot:** `Summarise what a DPU does in one sentence.`

**One-shot:**
```
Translate a Python concept into plain English.
Concept: list comprehension
Plain English: A shortcut for creating a list by applying a rule to each item.

Concept: dictionary
Plain English:
```

**Few-shot:**
```
Rate the code quality: GOOD, OKAY, or NEEDS WORK.
Code: x = 1 + 1        -> GOOD
Code: x=1+1            -> OKAY
Code: x=1+1;y=x*x;z=y  -> NEEDS WORK
Code: for i in range(n): total=total+i  ->
```

**Chain-of-Thought:** `If I sample the camera at 30 fps and each frame takes 5ms to process on the DPU, can I keep up in real time? Think step by step.`

**Role + Format + Constraint (all together):**
```
You are a code reviewer at a robotics company.
Review this Python snippet for a PYNQ project.
Format: 3 bullet points — one bug, one improvement, one compliment.
Keep each bullet to one sentence.

Code:
capture = cv2.VideoCapture(0)
frame = capture.read()
cv2.imshow('feed', frame)
```

## 6. Limits and Responsible Use

Language models are impressive, but they have real and important limitations. Understanding these is as important as knowing how to use the model effectively.

### 6.1 Hallucinations

A hallucination is when the model states something false with complete confidence. Because the model predicts likely-sounding text rather than looking up verified facts, it can generate plausible-sounding but entirely wrong answers.

Hallucinations happen regularly with:
- Obscure historical facts
- Specific numbers, dates, or statistics
- Paper citations and references (the model often invents titles and authors)
- Events after the knowledge cutoff

### 6.2 Knowledge cutoff

Training a model takes months and is done once. After training stops, the model's knowledge is frozen. It has no awareness of anything that happened afterward — new research, news events, software releases, or API changes.

### 6.3 Bias

The model learned from human-written text, which contains human biases — social, cultural, political. It can reproduce and amplify those biases in its outputs without any warning.

### The golden rule

> **Use AI as a smart first draft and thinking partner, not a source of truth. Always verify important information yourself.**

<div style="background:#141a24;border:1px solid #2f3b4d;border-radius:12px;padding:18px;font-family:sans-serif;"><div style="font-size:19px;font-weight:700;color:#38bdf8;margin-bottom:12px;">&#9888; AI can be confidently WRONG</div><table style="border-collapse:separate;border-spacing:10px 0;width:100%;table-layout:fixed;"><tr><td style="background:#1e2733;border-radius:10px;padding:12px;vertical-align:top;"><div style="font-size:20px;">&#127744;</div><div style="font-size:13px;font-weight:700;color:#f87171;margin:4px 0;">Hallucination</div><div style="font-size:11px;color:#94a3b8;">Makes up facts that<br>sound totally believable</div></td><td style="background:#1e2733;border-radius:10px;padding:12px;vertical-align:top;"><div style="font-size:20px;">&#128197;</div><div style="font-size:13px;font-weight:700;color:#60a5fa;margin:4px 0;">Knowledge Cutoff</div><div style="font-size:11px;color:#94a3b8;">Knows nothing after<br>its training date</div></td><td style="background:#1e2733;border-radius:10px;padding:12px;vertical-align:top;"><div style="font-size:20px;">&#9878;</div><div style="font-size:13px;font-weight:700;color:#c4b5fd;margin:4px 0;">Bias</div><div style="font-size:11px;color:#94a3b8;">Repeats biases from<br>the text it learned from</div></td></tr></table><div style="background:#10233b;border:1px solid #1e4976;border-radius:8px;padding:10px 14px;margin-top:10px;font-size:13px;color:#cfe3ff;"><b>&#127760; Real life:</b> Lawyers have been fined for submitting AI-written legal briefs that cited <b>court cases that never existed</b> &mdash; the AI invented them. Always verify what AI tells you!</div></div>

In [14]:
# ===== GAME: Fact or Fiction? (spot the hallucination) =====
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import random, threading, time

_Q = [
 {"c":"The PYNQ KV260 uses a Xilinx Zynq UltraScale+ MPSoC chip.","a":"fact",
  "e":"Correct! The KV260 is built on the Zynq UltraScale+ platform."},
 {"c":"Python was created by James Gosling in 1991.","a":"fiction",
  "e":"Gosling created Java. Python was made by Guido van Rossum. The AI mixed up two real facts!"},
 {"c":"GPT-4 has exactly 1.7 trillion parameters.","a":"fiction",
  "e":"OpenAI never confirmed this number. A classic hallucination: a precise-sounding but made-up stat."},
 {"c":"An FPGA can be reprogrammed after manufacturing, unlike an ASIC.","a":"fact",
  "e":"Correct! FPGAs are reconfigurable; ASICs are fixed once made."},
 {"c":"The DPU overlay runs at exactly 333 MHz with 4096 cores.","a":"fiction",
  "e":"Sounds technical, but these exact numbers are invented. That is how hallucinations trick you."},
 {"c":"The famous Battle of Cheeseburg in 1823 changed European trade.","a":"fiction",
  "e":"There was no Battle of Cheeseburg! But notice how believable it sounds."},
]
_qs = random.sample(_Q, len(_Q))
_i = [0]; _sc = [0,0]
_claim = widgets.HTML()
_bf = widgets.Button(description="Fact", button_style="success", layout=widgets.Layout(width="110px", height="40px"))
_bx = widgets.Button(description="Fiction", button_style="danger", layout=widgets.Layout(width="110px", height="40px"))
_res = widgets.Output(); _score = widgets.HTML()

def _show():
    i = _i[0]
    if i >= len(_qs):
        _claim.value = ("<div style='font-family:sans-serif;font-size:15px;color:#1e2a4a;padding:12px;'>"
                        f"&#127881; Done! Final score: {_sc[0]}/{_sc[1]}</div>")
        _bf.disabled = _bx.disabled = True; return
    q = _qs[i]
    _claim.value = (f"<div style='font-family:sans-serif;background:#f8faff;border:1px solid #e2e8f0;"
                    f"border-radius:10px;padding:16px;margin:8px 0;'>"
                    f"<div style='font-size:11px;color:#64748b;margin-bottom:6px;'>Claim {i+1} of {len(_qs)} "
                    f"&mdash; did the AI make this up?</div>"
                    f"<div style='font-size:15px;color:#1e2a4a;font-style:italic;'>\"{q['c']}\"</div></div>")
    _res.clear_output(); _bf.disabled = _bx.disabled = False

def _ans(choice):
    i = _i[0]
    if i >= len(_qs): return
    q = _qs[i]; ok = (choice == q["a"]); _sc[1]+=1
    if ok: _sc[0]+=1
    ic = "&#9989;" if ok else "&#10060;"; col = "#15803d" if ok else "#b91c1c"; bg = "#dcfce7" if ok else "#fee2e2"
    lbl = "Correct!" if ok else f"Nope &mdash; it was {q['a'].upper()}"
    with _res:
        clear_output(wait=True)
        display(HTML(f"<div style='font-family:sans-serif;background:{bg};border-radius:8px;padding:12px;'>"
                     f"<div style='font-weight:700;color:{col};font-size:14px;'>{ic} {lbl}</div>"
                     f"<div style='font-size:12px;color:#374151;margin-top:4px;'>{q['e']}</div></div>"))
    _score.value = f"<div style='font-family:sans-serif;font-size:12px;color:#64748b;'>Score: {_sc[0]}/{_sc[1]}</div>"
    _bf.disabled = _bx.disabled = True; _i[0]+=1
    def _nxt(): time.sleep(2.5); _show()
    threading.Thread(target=_nxt, daemon=True).start()

_bf.on_click(lambda _: _ans("fact")); _bx.on_click(lambda _: _ans("fiction"))
display(HTML("<div style='font-family:sans-serif;font-size:16px;font-weight:700;color:#1e2a4a;'>"
             "&#127918; GAME: Fact or Fiction?</div>"
             "<div style='font-size:13px;color:#64748b;margin-bottom:8px;'>Each claim was written by an AI. "
             "Can you spot the hallucinations?</div>"))
display(_claim); display(widgets.HBox([_bf, _bx])); display(_res); display(_score); _show()


HTML(value='')

Output()

HTML(value='')

## 7. Switching Models

The Strix is running three models of different sizes. Larger models are generally more capable but slower — the capability-versus-speed tradeoff is fundamental to real-world AI deployment.

| Friendly Name | Real Model | Parameters | Best For |
| --- | --- | --- | --- |
| Quick Helper | Qwen3-VL-8B | 8 billion | Fast answers, general questions, image input |
| Code Helper | Qwen3-Coder-30B | 30 billion | Python, debugging, technical explanations |
| Smart Helper | Qwen3.6-27B | 27 billion | Complex reasoning, detailed analysis |

You can switch models using the dropdown in the chat panel

Let us test it with prompt :
`Write a Python function that reverses a string. Include a docstring and a test.`

<div style="background:#141a24;border:1px solid #2f3b4d;border-radius:12px;padding:18px;font-family:sans-serif;"><div style="font-size:19px;font-weight:700;color:#38bdf8;margin-bottom:12px;">&#9878; Bigger model = smarter but slower</div><table style="border-collapse:separate;border-spacing:10px 0;width:100%;table-layout:fixed;"><tr><td style="background:#1e2733;border-radius:10px;padding:12px;vertical-align:top;"><div style="font-size:14px;font-weight:700;color:#4ade80;margin-bottom:6px;">Quick Helper (8B)</div><div style="font-size:11px;color:#94a3b8;">Fast &#9889; sees images<br>Good for most questions</div></td><td style="background:#1e2733;border-radius:10px;padding:12px;vertical-align:top;"><div style="font-size:14px;font-weight:700;color:#fbbf24;margin-bottom:6px;">Code Helper (30B)</div><div style="font-size:11px;color:#94a3b8;">Slower, best at Python<br>Debugging and code</div></td><td style="background:#1e2733;border-radius:10px;padding:12px;vertical-align:top;"><div style="font-size:14px;font-weight:700;color:#a78bfa;margin-bottom:6px;">Smart Helper (27B)</div><div style="font-size:11px;color:#94a3b8;">Slowest, deep reasoning<br>Hard problems</div></td></tr></table><div style="background:#10233b;border:1px solid #1e4976;border-radius:8px;padding:10px 14px;margin-top:10px;font-size:13px;color:#cfe3ff;"><b>&#127760; Real life:</b> This is the same trade-off phone makers face &mdash; a bigger AI model gives better answers but drains the battery and runs hotter. Picking the right size is real engineering.</div></div>

In [ ]:
# launch chat 
launch_chat()

**Note:** If the 30B model is slow or unavailable, that is normal during a busy session — many students sharing the same GPU reduces throughput. Switch back to Quick Helper for interactive work.

## 8. You Made It!

Great work getting through this session. Language models are one of the most powerful tools in modern software, and understanding how they actually work — tokens, context, probability, sampling — puts you well ahead of most people who just use them as a black box.

Don't forget to stop the chat server when you are done to free up the port for other sessions!

| Concept | Key Idea |
| --- | --- |
| LLM | A neural network that predicts the next token, one at a time |
| Parameters | The billions of tuned numbers that store the model's learned patterns |
| Tokens | The chunks the model reads; ~¾ of an English word each |
| Context | A list of messages re-sent every turn — the AI's working memory |
| Zero-Shot | Ask directly with no examples; relies on pre-trained knowledge |
| One-Shot | One example sets the pattern for the model to follow |
| Few-Shot | Multiple examples for format control and consistent outputs |
| Chain-of-Thought | Ask the model to reason step by step before answering |
| Role Prompting | Define a persona via system instructions to control tone and behaviour |
| Hallucination | The model confidently stating false things — always verify |

In [ ]:
# Stop the chat server to free up the port for other sessions.
from bootcamp_ai import stop_chat
stop_chat()